# Modelo de aprendizaje automático para brillo y cinemática residual de las Gemínidas

In [1]:
import hashlib
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    PolynomialFeatures,
    StandardScaler,
)


## Etapa 11. Preparación del modelado y diseño de la validación temporal

### Carga y auditoría de las bases de modelado

In [2]:
# Importar las dependencias y comprobar las rutas de los archivos de modelado

ruta_actual = Path.cwd().resolve()
raiz_proyecto = next((ruta for ruta in [ruta_actual, *ruta_actual.parents] if (ruta / "pyproject.toml").is_file()), None)

if raiz_proyecto is None:
    raise FileNotFoundError("No se encontró la raíz del proyecto a partir del directorio actual.")

carpeta_modelado = raiz_proyecto / "data" / "processed" / "geminidas_modelado_v2"
ruta_entrenamiento = carpeta_modelado / "geminidas_modelado_entrenamiento_2019_2024_v2.parquet"
ruta_prueba = carpeta_modelado / "geminidas_modelado_prueba_2025_v2.parquet"
ruta_diccionario = carpeta_modelado / "metadata" / "diccionario_variables_modelado_v2.csv"
ruta_configuracion = carpeta_modelado / "metadata" / "configuracion_modelado_v2.csv"

rutas_entrada = {
    "Entrenamiento": ruta_entrenamiento,
    "Prueba": ruta_prueba,
    "Diccionario": ruta_diccionario,
    "Configuración": ruta_configuracion,
}

archivos_faltantes = [str(ruta) for ruta in rutas_entrada.values() if not ruta.is_file()]

if archivos_faltantes:
    raise FileNotFoundError("No se encontraron los siguientes archivos:\n" + "\n".join(archivos_faltantes))

pd.DataFrame(
    {
        "archivo": rutas_entrada.keys(),
        "ruta": [str(ruta.relative_to(raiz_proyecto)) for ruta in rutas_entrada.values()],
        "existe": [ruta.is_file() for ruta in rutas_entrada.values()],
    }
)

,archivo,ruta,existe
0,Entrenamiento,data\processed\geminidas_modelado_v2\geminidas...,True
1,Prueba,data\processed\geminidas_modelado_v2\geminidas...,True
2,Diccionario,data\processed\geminidas_modelado_v2\metadata\...,True
3,Configuración,data\processed\geminidas_modelado_v2\metadata\...,True


In [3]:
# Cargar las bases y auditar su estructura, documentación y separación temporal

df_entrenamiento = pd.read_parquet(ruta_entrenamiento)
df_prueba = pd.read_parquet(ruta_prueba)
df_diccionario = pd.read_csv(ruta_diccionario)
df_configuracion = pd.read_csv(ruta_configuracion)

variables_documentadas = set(df_diccionario["variable"])
columnas_entrenamiento = set(df_entrenamiento.columns)
columnas_prueba = set(df_prueba.columns)
identificadores_entrenamiento = set(df_entrenamiento["id_fila_origen"])
identificadores_prueba = set(df_prueba["id_fila_origen"])

resumen_bases = pd.DataFrame(
    [
        {
            "base": "Entrenamiento",
            "filas": len(df_entrenamiento),
            "columnas": df_entrenamiento.shape[1],
            "anio_minimo": df_entrenamiento["source_year"].min(),
            "anio_maximo": df_entrenamiento["source_year"].max(),
            "identificadores_unicos": df_entrenamiento["id_fila_origen"].is_unique,
        },
        {
            "base": "Prueba",
            "filas": len(df_prueba),
            "columnas": df_prueba.shape[1],
            "anio_minimo": df_prueba["source_year"].min(),
            "anio_maximo": df_prueba["source_year"].max(),
            "identificadores_unicos": df_prueba["id_fila_origen"].is_unique,
        },
    ]
)

comprobaciones_estructura = pd.DataFrame(
    [
        {"comprobacion": "Filas de entrenamiento correctas", "resultado": len(df_entrenamiento) == 12904},
        {"comprobacion": "Filas de prueba correctas", "resultado": len(df_prueba) == 6175},
        {"comprobacion": "Ambas bases tienen las mismas columnas", "resultado": columnas_entrenamiento == columnas_prueba},
        {"comprobacion": "Todas las columnas están documentadas", "resultado": columnas_entrenamiento == variables_documentadas},
        {"comprobacion": "Identificadores de entrenamiento únicos", "resultado": df_entrenamiento["id_fila_origen"].is_unique},
        {"comprobacion": "Identificadores de prueba únicos", "resultado": df_prueba["id_fila_origen"].is_unique},
        {"comprobacion": "Sin identificadores compartidos", "resultado": identificadores_entrenamiento.isdisjoint(identificadores_prueba)},
        {"comprobacion": "Entrenamiento contiene 2019–2024", "resultado": set(df_entrenamiento["source_year"].unique()) == set(range(2019, 2025))},
        {"comprobacion": "Prueba contiene solamente 2025", "resultado": set(df_prueba["source_year"].unique()) == {2025}},
        {"comprobacion": "Partición de entrenamiento correcta", "resultado": set(df_entrenamiento["particion_modelado"].unique()) == {"entrenamiento"}},
        {"comprobacion": "Partición de prueba correcta", "resultado": set(df_prueba["particion_modelado"].unique()) == {"prueba"}},
    ]
)

display(resumen_bases)
display(comprobaciones_estructura)

if not comprobaciones_estructura["resultado"].all():
    raise ValueError("La auditoría estructural detectó al menos una inconsistencia.")

,base,filas,columnas,anio_minimo,anio_maximo,identificadores_unicos
0,Entrenamiento,12904,39,2019,2024,True
1,Prueba,6175,39,2025,2025,True


,comprobacion,resultado
0,Filas de entrenamiento correctas,True
1,Filas de prueba correctas,True
2,Ambas bases tienen las mismas columnas,True
3,Todas las columnas están documentadas,True
4,Identificadores de entrenamiento únicos,True
5,Identificadores de prueba únicos,True
6,Sin identificadores compartidos,True
7,Entrenamiento contiene 2019–2024,True
8,Prueba contiene solamente 2025,True
9,Partición de entrenamiento correcta,True


### Construcción de las muestras utilizables para el modelo

In [4]:
# Construir las muestras estándar y estricta utilizables para el modelado

columnas_seleccion_ml = ["id_fila_origen", "usar_modelo_ml", "objetivo_ml", "grupo_brillo_ml", "pertenece_muestra_estricta"]
columnas_ausentes_ml = [columna for columna in columnas_seleccion_ml if columna not in df_entrenamiento.columns]

if columnas_ausentes_ml:
    raise KeyError(f"Faltan columnas necesarias para construir las muestras de ML: {columnas_ausentes_ml}")

df_entrenamiento_ml = df_entrenamiento.loc[df_entrenamiento["usar_modelo_ml"]].copy()
df_prueba_ml = df_prueba.loc[df_prueba["usar_modelo_ml"]].copy()
df_entrenamiento_ml_estricto = df_entrenamiento_ml.loc[df_entrenamiento_ml["pertenece_muestra_estricta"]].copy()
df_prueba_ml_estricto = df_prueba_ml.loc[df_prueba_ml["pertenece_muestra_estricta"]].copy()

resumen_muestras_ml = pd.DataFrame(
    [
        {
            "particion": "Entrenamiento",
            "muestra": "Estándar",
            "trayectorias_base": len(df_entrenamiento),
            "trayectorias_utilizables_ml": len(df_entrenamiento_ml),
            "porcentaje_utilizable": 100 * len(df_entrenamiento_ml) / len(df_entrenamiento),
            "objetivos_faltantes": df_entrenamiento_ml["objetivo_ml"].isna().sum(),
        },
        {
            "particion": "Entrenamiento",
            "muestra": "Estricta",
            "trayectorias_base": df_entrenamiento["pertenece_muestra_estricta"].sum(),
            "trayectorias_utilizables_ml": len(df_entrenamiento_ml_estricto),
            "porcentaje_utilizable": 100 * len(df_entrenamiento_ml_estricto) / df_entrenamiento["pertenece_muestra_estricta"].sum(),
            "objetivos_faltantes": df_entrenamiento_ml_estricto["objetivo_ml"].isna().sum(),
        },
        {
            "particion": "Prueba",
            "muestra": "Estándar",
            "trayectorias_base": len(df_prueba),
            "trayectorias_utilizables_ml": len(df_prueba_ml),
            "porcentaje_utilizable": 100 * len(df_prueba_ml) / len(df_prueba),
            "objetivos_faltantes": df_prueba_ml["objetivo_ml"].isna().sum(),
        },
        {
            "particion": "Prueba",
            "muestra": "Estricta",
            "trayectorias_base": df_prueba["pertenece_muestra_estricta"].sum(),
            "trayectorias_utilizables_ml": len(df_prueba_ml_estricto),
            "porcentaje_utilizable": 100 * len(df_prueba_ml_estricto) / df_prueba["pertenece_muestra_estricta"].sum(),
            "objetivos_faltantes": df_prueba_ml_estricto["objetivo_ml"].isna().sum(),
        },
    ]
)

comprobaciones_muestras_ml = pd.DataFrame(
    [
        {"comprobacion": "Entrenamiento estándar conserva 8,632 casos", "resultado": len(df_entrenamiento_ml) == 8632},
        {"comprobacion": "Entrenamiento estricto conserva 2,727 casos", "resultado": len(df_entrenamiento_ml_estricto) == 2727},
        {"comprobacion": "Prueba estándar conserva 4,182 casos", "resultado": len(df_prueba_ml) == 4182},
        {"comprobacion": "Prueba estricta conserva 1,943 casos", "resultado": len(df_prueba_ml_estricto) == 1943},
        {"comprobacion": "Objetivo completo en entrenamiento", "resultado": df_entrenamiento_ml["objetivo_ml"].notna().all()},
        {"comprobacion": "Objetivo binario en entrenamiento", "resultado": set(df_entrenamiento_ml["objetivo_ml"].unique()) == {0, 1}},
        {"comprobacion": "Identificadores únicos en entrenamiento ML", "resultado": df_entrenamiento_ml["id_fila_origen"].is_unique},
        {"comprobacion": "Identificadores únicos en prueba ML", "resultado": df_prueba_ml["id_fila_origen"].is_unique},
    ]
)

display(resumen_muestras_ml.round({"porcentaje_utilizable": 2}))
display(comprobaciones_muestras_ml)

if not comprobaciones_muestras_ml["resultado"].all():
    raise ValueError("La construcción de las muestras de modelado detectó alguna inconsistencia.")

,particion,muestra,trayectorias_base,trayectorias_utilizables_ml,porcentaje_utilizable,objetivos_faltantes
0,Entrenamiento,Estándar,12904,8632,66.89,0
1,Entrenamiento,Estricta,4166,2727,65.46,0
2,Prueba,Estándar,6175,4182,67.72,0
3,Prueba,Estricta,2874,1943,67.61,0


,comprobacion,resultado
0,"Entrenamiento estándar conserva 8,632 casos",True
1,"Entrenamiento estricto conserva 2,727 casos",True
2,"Prueba estándar conserva 4,182 casos",True
3,"Prueba estricta conserva 1,943 casos",True
4,Objetivo completo en entrenamiento,True
5,Objetivo binario en entrenamiento,True
6,Identificadores únicos en entrenamiento ML,True
7,Identificadores únicos en prueba ML,True


### Definición y auditoría del objetivo binario

In [5]:
# Auditar la codificación, los límites y el balance del objetivo binario de entrenamiento

configuracion_modelado = df_configuracion.set_index("elemento")["valor"]
limite_brillante = float(configuracion_modelado.loc["Límite brillante"])
limite_tenue = float(configuracion_modelado.loc["Límite tenue"])

y_entrenamiento = df_entrenamiento_ml["objetivo_ml"].astype(int)
y_entrenamiento_estricto = df_entrenamiento_ml_estricto["objetivo_ml"].astype(int)

resumen_objetivo = (
    df_entrenamiento_ml.groupby("objetivo_ml", observed=True)
    .agg(
        trayectorias=("id_fila_origen", "size"),
        magnitud_minima=("magnitud_analisis", "min"),
        magnitud_mediana=("magnitud_analisis", "median"),
        magnitud_maxima=("magnitud_analisis", "max"),
    )
    .reset_index()
)

resumen_objetivo["clase"] = resumen_objetivo["objetivo_ml"].map({0: "Brillante", 1: "Tenue"})
resumen_objetivo["porcentaje_entrenamiento"] = 100 * resumen_objetivo["trayectorias"] / len(df_entrenamiento_ml)
resumen_objetivo = resumen_objetivo[["objetivo_ml", "clase", "trayectorias", "porcentaje_entrenamiento", "magnitud_minima", "magnitud_mediana", "magnitud_maxima"]]

conteos_estandar = y_entrenamiento.value_counts()
conteos_estrictos = y_entrenamiento_estricto.value_counts()

comprobaciones_objetivo = pd.DataFrame(
    [
        {"comprobacion": "Codificación binaria completa", "resultado": set(y_entrenamiento.unique()) == {0, 1}},
        {"comprobacion": "Clase brillante respeta el límite", "resultado": df_entrenamiento_ml.loc[y_entrenamiento == 0, "magnitud_analisis"].le(limite_brillante).all()},
        {"comprobacion": "Clase tenue respeta el límite", "resultado": df_entrenamiento_ml.loc[y_entrenamiento == 1, "magnitud_analisis"].ge(limite_tenue).all()},
        {"comprobacion": "Estándar contiene 4,300 brillantes", "resultado": conteos_estandar.get(0, 0) == 4300},
        {"comprobacion": "Estándar contiene 4,332 tenues", "resultado": conteos_estandar.get(1, 0) == 4332},
        {"comprobacion": "Estricta contiene 1,681 brillantes", "resultado": conteos_estrictos.get(0, 0) == 1681},
        {"comprobacion": "Estricta contiene 1,046 tenues", "resultado": conteos_estrictos.get(1, 0) == 1046},
    ]
)

print(f"Límite brillante: {limite_brillante:.2f}")
print(f"Límite tenue: {limite_tenue:.2f}")
display(resumen_objetivo.round(4))
display(comprobaciones_objetivo)

if not comprobaciones_objetivo["resultado"].all():
    raise ValueError("La auditoría del objetivo binario detectó alguna inconsistencia.")

Límite brillante: -0.89
Límite tenue: -0.07


,objetivo_ml,clase,trayectorias,porcentaje_entrenamiento,magnitud_minima,magnitud_mediana,magnitud_maxima
0,0,Brillante,4300,49.8146,-7.64,-1.38,-0.89
1,1,Tenue,4332,50.1854,-0.07,0.44,3.82


,comprobacion,resultado
0,Codificación binaria completa,True
1,Clase brillante respeta el límite,True
2,Clase tenue respeta el límite,True
3,"Estándar contiene 4,300 brillantes",True
4,"Estándar contiene 4,332 tenues",True
5,"Estricta contiene 1,681 brillantes",True
6,"Estricta contiene 1,046 tenues",True


### Definición y auditoría de los bloques de predictores

In [6]:
# Definir los bloques de predictores y comprobar su integridad numérica

variables_fase = ["delta_sol_lon_deg"]
variables_calidad = ["Qc (deg)", "MedianFitErr (arcsec)", "numero_estaciones", "sigma_radiante_aprox", "Vgeo_sigma_modelo"]
variables_cinematica = ["delta_v_paralela_km_s", "delta_v_lambda_km_s", "delta_v_beta_km_s"]
variables_validacion = ["source_year"]
variables_pca_futuras = ["PC1", "PC2", "PC3"]

bloques_predictores = {
    "Fase solar": variables_fase,
    "Calidad observacional": variables_calidad,
    "Cinemática residual local": variables_cinematica,
}

variables_requeridas = variables_fase + variables_calidad + variables_cinematica + variables_validacion
variables_ausentes = [variable for variable in variables_requeridas if variable not in df_entrenamiento_ml.columns]

if variables_ausentes:
    raise KeyError(f"Faltan variables necesarias para el modelado: {variables_ausentes}")

filas_resumen_entrenamiento = []

for bloque, variables in bloques_predictores.items():
    for variable in variables:
        valores = pd.to_numeric(df_entrenamiento_ml[variable], errors="coerce")
        filas_resumen_entrenamiento.append(
            {
                "bloque": bloque,
                "variable": variable,
                "faltantes": valores.isna().sum(),
                "no_finitos": (~np.isfinite(valores.dropna())).sum(),
                "minimo": valores.min(),
                "mediana": valores.median(),
                "maximo": valores.max(),
            }
        )

resumen_predictores_entrenamiento = pd.DataFrame(filas_resumen_entrenamiento)

auditoria_particiones = pd.DataFrame(
    [
        {
            "particion": nombre,
            "trayectorias": len(base),
            "faltantes_predictores": base[variables_requeridas].isna().sum().sum(),
            "valores_no_finitos": sum((~np.isfinite(pd.to_numeric(base[variable], errors="coerce").dropna())).sum() for variable in variables_requeridas),
        }
        for nombre, base in [("Entrenamiento", df_entrenamiento_ml), ("Prueba", df_prueba_ml)]
    ]
)

display(resumen_predictores_entrenamiento.round(6))
display(auditoria_particiones)

if resumen_predictores_entrenamiento[["faltantes", "no_finitos"]].to_numpy().sum() != 0:
    raise ValueError("Los predictores de entrenamiento contienen valores faltantes o no finitos.")

if auditoria_particiones[["faltantes_predictores", "valores_no_finitos"]].to_numpy().sum() != 0:
    raise ValueError("Alguna partición contiene predictores faltantes o no finitos.")

,bloque,variable,faltantes,no_finitos,minimo,mediana,maximo
0,Fase solar,delta_sol_lon_deg,0,0,-4.990121,-0.307996,1.983771
1,Calidad observacional,Qc (deg),0,0,15.080000,65.935000,90.000000
2,Calidad observacional,MedianFitErr (arcsec),0,0,2.730000,43.565000,60.000000
3,Calidad observacional,numero_estaciones,0,0,2.000000,3.000000,20.000000
4,Calidad observacional,sigma_radiante_aprox,0,0,0.009896,0.161504,0.249968
5,Calidad observacional,Vgeo_sigma_modelo,0,0,0.000500,0.060200,2.130500
6,Cinemática residual local,delta_v_paralela_km_s,0,0,-3.478542,-0.008750,3.471391
7,Cinemática residual local,delta_v_lambda_km_s,0,0,-1.245482,0.010680,1.192395
8,Cinemática residual local,delta_v_beta_km_s,0,0,-1.179222,0.007675,1.096174


,particion,trayectorias,faltantes_predictores,valores_no_finitos
0,Entrenamiento,8632,0,0
1,Prueba,4182,0,0


### Diseño de la validación temporal interna

In [7]:
# Construir y auditar los pliegues de validación temporal creciente

def construir_pliegues_temporales(df_modelado, nombre_muestra):
    anios = sorted(df_modelado["source_year"].unique())
    pliegues = []
    filas_resumen = []

    for anio_validacion in anios[1:]:
        mascara_entrenamiento = df_modelado["source_year"] < anio_validacion
        mascara_validacion = df_modelado["source_year"] == anio_validacion
        indices_entrenamiento = np.flatnonzero(mascara_entrenamiento.to_numpy())
        indices_validacion = np.flatnonzero(mascara_validacion.to_numpy())
        objetivo_entrenamiento = df_modelado.iloc[indices_entrenamiento]["objetivo_ml"].astype(int)
        objetivo_validacion = df_modelado.iloc[indices_validacion]["objetivo_ml"].astype(int)

        pliegues.append((indices_entrenamiento, indices_validacion))
        filas_resumen.append(
            {
                "muestra": nombre_muestra,
                "pliegue": len(pliegues),
                "anios_entrenamiento": f"{df_modelado.iloc[indices_entrenamiento]['source_year'].min()}–{df_modelado.iloc[indices_entrenamiento]['source_year'].max()}",
                "anio_validacion": anio_validacion,
                "entrenamiento_total": len(indices_entrenamiento),
                "entrenamiento_brillantes": (objetivo_entrenamiento == 0).sum(),
                "entrenamiento_tenues": (objetivo_entrenamiento == 1).sum(),
                "validacion_total": len(indices_validacion),
                "validacion_brillantes": (objetivo_validacion == 0).sum(),
                "validacion_tenues": (objetivo_validacion == 1).sum(),
            }
        )

    return pliegues, pd.DataFrame(filas_resumen)

pliegues_temporales_estandar, resumen_pliegues_estandar = construir_pliegues_temporales(df_entrenamiento_ml, "Estándar")
pliegues_temporales_estrictos, resumen_pliegues_estrictos = construir_pliegues_temporales(df_entrenamiento_ml_estricto, "Estricta")
resumen_pliegues_temporales = pd.concat([resumen_pliegues_estandar, resumen_pliegues_estrictos], ignore_index=True)

comprobaciones_pliegues = pd.DataFrame(
    [
        {"comprobacion": "Cinco pliegues estándar", "resultado": len(pliegues_temporales_estandar) == 5},
        {"comprobacion": "Cinco pliegues estrictos", "resultado": len(pliegues_temporales_estrictos) == 5},
        {"comprobacion": "Ambas clases presentes en cada entrenamiento", "resultado": (resumen_pliegues_temporales[["entrenamiento_brillantes", "entrenamiento_tenues"]] > 0).all().all()},
        {"comprobacion": "Ambas clases presentes en cada validación", "resultado": (resumen_pliegues_temporales[["validacion_brillantes", "validacion_tenues"]] > 0).all().all()},
        {"comprobacion": "2025 permanece fuera de la validación interna", "resultado": resumen_pliegues_temporales["anio_validacion"].max() == 2024},
    ]
)

display(resumen_pliegues_temporales)
display(comprobaciones_pliegues)

if not comprobaciones_pliegues["resultado"].all():
    raise ValueError("El esquema de validación temporal no satisface todas las condiciones.")

,muestra,pliegue,anios_entrenamiento,anio_validacion,entrenamiento_total,entrenamiento_brillantes,entrenamiento_tenues,validacion_total,validacion_brillantes,validacion_tenues
0,Estándar,1,2019–2019,2020,274,131,143,703,319,384
1,Estándar,2,2019–2020,2021,977,450,527,1319,437,882
2,Estándar,3,2019–2021,2022,2296,887,1409,2433,1405,1028
3,Estándar,4,2019–2022,2023,4729,2292,2437,2258,997,1261
4,Estándar,5,2019–2023,2024,6987,3289,3698,1645,1011,634
5,Estricta,1,2019–2019,2020,58,39,19,116,56,60
6,Estricta,2,2019–2020,2021,174,95,79,192,61,131
7,Estricta,3,2019–2021,2022,366,156,210,1153,799,354
8,Estricta,4,2019–2022,2023,1519,955,564,824,423,401
9,Estricta,5,2019–2023,2024,2343,1378,965,384,303,81


,comprobacion,resultado
0,Cinco pliegues estándar,True
1,Cinco pliegues estrictos,True
2,Ambas clases presentes en cada entrenamiento,True
3,Ambas clases presentes en cada validación,True
4,2025 permanece fuera de la validación interna,True


### Definición del preprocesamiento dentro de cada pliegue

In [8]:
# Definir el preprocesamiento de fase, calidad y cinemática sin ajustarlo todavía

transformador_fase = Pipeline(
    [
        ("terminos_fase", PolynomialFeatures(degree=2, include_bias=False)),
        ("escalamiento_fase", StandardScaler()),
    ]
)

transformador_calidad = Pipeline(
    [
        ("escalamiento_calidad", StandardScaler()),
    ]
)

transformador_cinematica = Pipeline(
    [
        ("escalamiento_cinematica", StandardScaler()),
    ]
)

preprocesador_control = ColumnTransformer(
    [
        ("fase", transformador_fase, variables_fase),
        ("calidad", transformador_calidad, variables_calidad),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

preprocesador_control_cinematica = ColumnTransformer(
    [
        ("fase", transformador_fase, variables_fase),
        ("calidad", transformador_calidad, variables_calidad),
        ("cinematica", transformador_cinematica, variables_cinematica),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

configuracion_preprocesamiento = pd.DataFrame(
    [
        {
            "bloque": "Fase solar",
            "variables_originales": ", ".join(variables_fase),
            "transformacion": "Término lineal + término cuadrático + estandarización",
            "numero_salidas": 2,
        },
        {
            "bloque": "Calidad observacional",
            "variables_originales": ", ".join(variables_calidad),
            "transformacion": "Estandarización",
            "numero_salidas": len(variables_calidad),
        },
        {
            "bloque": "Cinemática residual local",
            "variables_originales": ", ".join(variables_cinematica),
            "transformacion": "Estandarización",
            "numero_salidas": len(variables_cinematica),
        },
    ]
)

configuracion_modelos_iniciales = pd.DataFrame(
    [
        {
            "modelo": "Control",
            "fase": True,
            "calidad": True,
            "cinematica": False,
            "numero_predictores_transformados": 2 + len(variables_calidad),
        },
        {
            "modelo": "Control + cinemática local",
            "fase": True,
            "calidad": True,
            "cinematica": True,
            "numero_predictores_transformados": 2 + len(variables_calidad) + len(variables_cinematica),
        },
    ]
)

display(configuracion_preprocesamiento)
display(configuracion_modelos_iniciales)

if hasattr(preprocesador_control, "transformers_") or hasattr(preprocesador_control_cinematica, "transformers_"):
    raise RuntimeError("Los preprocesadores no deben estar ajustados antes de la validación temporal.")

,bloque,variables_originales,transformacion,numero_salidas
0,Fase solar,delta_sol_lon_deg,Término lineal + término cuadrático + estandar...,2
1,Calidad observacional,"Qc (deg), MedianFitErr (arcsec), numero_estaci...",Estandarización,5
2,Cinemática residual local,"delta_v_paralela_km_s, delta_v_lambda_km_s, de...",Estandarización,3


,modelo,fase,calidad,cinematica,numero_predictores_transformados
0,Control,True,True,False,7
1,Control + cinemática local,True,True,True,10


## Etapa 12. Entrenamiento y comparación temporal de los modelos

### Modelo de control con fase solar y calidad observacional

In [9]:
# Entrenar y evaluar temporalmente el modelo de control mediante regresión logística

def evaluar_modelo_temporal(modelo, nombre_modelo, df_modelado, pliegues, preparar_pliegue=None):
    filas_metricas = []
    tablas_predicciones = []

    for numero_pliegue, (indices_entrenamiento, indices_validacion) in enumerate(pliegues, start=1):
        modelo_pliegue = clone(modelo)
        X_entrenamiento = df_modelado.iloc[indices_entrenamiento]
        y_entrenamiento_pliegue = X_entrenamiento["objetivo_ml"].astype(int)
        X_validacion = df_modelado.iloc[indices_validacion]
        y_validacion = X_validacion["objetivo_ml"].astype(int)
        anio_validacion = int(X_validacion["source_year"].iloc[0])
        if preparar_pliegue is not None:
            X_entrenamiento, X_validacion = preparar_pliegue(
                X_entrenamiento.copy(),
                X_validacion.copy(),
                anio_validacion,
            )

        modelo_pliegue.fit(X_entrenamiento, y_entrenamiento_pliegue)

        probabilidades = modelo_pliegue.predict_proba(X_validacion)[:, 1]
        predicciones = (probabilidades >= 0.5).astype(int)

        tabla_predicciones = pd.DataFrame(
            {
                "id_fila_origen": X_validacion["id_fila_origen"].to_numpy(),
                "modelo": nombre_modelo,
                "pliegue": numero_pliegue,
                "anio_validacion": anio_validacion,
                "objetivo_real": y_validacion.to_numpy(),
                "probabilidad_tenue": probabilidades,
                "prediccion": predicciones,
                "pertenece_muestra_estricta": X_validacion["pertenece_muestra_estricta"].to_numpy(),
            }
        )

        tablas_predicciones.append(tabla_predicciones)

        for nombre_muestra, mascara in [
            ("Estándar", np.ones(len(X_validacion), dtype=bool)),
            ("Estricta", X_validacion["pertenece_muestra_estricta"].to_numpy(dtype=bool)),
        ]:
            y_real_muestra = y_validacion.to_numpy()[mascara]
            probabilidades_muestra = probabilidades[mascara]
            predicciones_muestra = predicciones[mascara]
            verdaderos_brillantes, falsos_tenues, falsos_brillantes, verdaderos_tenues = confusion_matrix(y_real_muestra, predicciones_muestra, labels=[0, 1]).ravel()

            filas_metricas.append(
                {
                    "modelo": nombre_modelo,
                    "muestra": nombre_muestra,
                    "pliegue": numero_pliegue,
                    "anio_validacion": anio_validacion,
                    "trayectorias": len(y_real_muestra),
                    "brillantes": (y_real_muestra == 0).sum(),
                    "tenues": (y_real_muestra == 1).sum(),
                    "roc_auc": roc_auc_score(y_real_muestra, probabilidades_muestra),
                    "average_precision": average_precision_score(y_real_muestra, probabilidades_muestra),
                    "exactitud": accuracy_score(y_real_muestra, predicciones_muestra),
                    "exactitud_balanceada": balanced_accuracy_score(y_real_muestra, predicciones_muestra),
                    "sensibilidad_tenues": recall_score(y_real_muestra, predicciones_muestra, pos_label=1),
                    "especificidad_brillantes": recall_score(y_real_muestra, predicciones_muestra, pos_label=0),
                    "verdaderos_brillantes": verdaderos_brillantes,
                    "falsos_tenues": falsos_tenues,
                    "falsos_brillantes": falsos_brillantes,
                    "verdaderos_tenues": verdaderos_tenues,
                }
            )

    return pd.DataFrame(filas_metricas), pd.concat(tablas_predicciones, ignore_index=True)

modelo_control = Pipeline(
    [
        ("preprocesamiento", preprocesador_control),
        ("clasificador", LogisticRegression(C=1.0, l1_ratio=0.0, solver="lbfgs", max_iter=5000)),
    ]
)

metricas_control, predicciones_control = evaluar_modelo_temporal(modelo_control, "Control", df_entrenamiento_ml, pliegues_temporales_estandar)

columnas_metricas = ["roc_auc", "average_precision", "exactitud", "exactitud_balanceada", "sensibilidad_tenues", "especificidad_brillantes"]
resumen_control = metricas_control.groupby(["modelo", "muestra"])[columnas_metricas].agg(["mean", "std"]).reset_index()
resumen_control.columns = ["modelo", "muestra"] + [f"{metrica}_{estadistico}" for metrica, estadistico in resumen_control.columns.tolist()[2:]]

display(metricas_control.round(4))
display(resumen_control.round(4))

if len(predicciones_control) != sum(len(indices_validacion) for _, indices_validacion in pliegues_temporales_estandar):
    raise ValueError("No se conservaron todas las predicciones temporales esperadas.")

,modelo,muestra,pliegue,anio_validacion,trayectorias,brillantes,tenues,roc_auc,average_precision,exactitud,exactitud_balanceada,sensibilidad_tenues,especificidad_brillantes,verdaderos_brillantes,falsos_tenues,falsos_brillantes,verdaderos_tenues
0,Control,Estándar,1,2020,703,319,384,0.7496,0.7995,0.6728,0.6666,0.7344,0.5987,191,128,102,282
1,Control,Estricta,1,2020,116,56,60,0.8104,0.7867,0.6293,0.6393,0.3500,0.9286,52,4,39,21
2,Control,Estándar,2,2021,1319,437,882,0.8172,0.8904,0.7612,0.7296,0.8231,0.6362,278,159,156,726
3,Control,Estricta,2,2021,192,61,131,0.7686,0.8618,0.6927,0.7266,0.6336,0.8197,50,11,48,83
4,Control,Estándar,3,2022,2433,1405,1028,0.6726,0.6200,0.6108,0.6224,0.6975,0.5473,769,636,311,717
5,Control,Estricta,3,2022,1153,799,354,0.6124,0.4440,0.5716,0.5768,0.5904,0.5632,450,349,145,209
6,Control,Estándar,4,2023,2258,997,1261,0.7426,0.7741,0.6634,0.6754,0.5726,0.7783,776,221,539,722
7,Control,Estricta,4,2023,824,423,401,0.7452,0.7373,0.6383,0.6307,0.3441,0.9173,388,35,263,138
8,Control,Estándar,5,2024,1645,1011,634,0.7918,0.7238,0.7052,0.7157,0.7618,0.6696,677,334,151,483
9,Control,Estricta,5,2024,384,303,81,0.6859,0.4186,0.7734,0.6393,0.4074,0.8713,264,39,48,33


,modelo,muestra,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,exactitud_mean,exactitud_std,exactitud_balanceada_mean,exactitud_balanceada_std,sensibilidad_tenues_mean,sensibilidad_tenues_std,especificidad_brillantes_mean,especificidad_brillantes_std
0,Control,Estricta,0.7245,0.0771,0.6497,0.2044,0.6611,0.0761,0.6426,0.0537,0.4651,0.1372,0.820,0.1499
1,Control,Estándar,0.7548,0.0552,0.7616,0.0996,0.6827,0.0555,0.6820,0.0425,0.7179,0.0933,0.646,0.0868


### 12.2 Modelo con componentes cinemáticas locales lineales

#### 12.2.1 Versión lineal

In [10]:
# Entrenar el modelo cinemático lineal y comparar su desempeño con el modelo de control

modelo_cinematica_lineal = Pipeline(
    [
        ("preprocesamiento", preprocesador_control_cinematica),
        ("clasificador", LogisticRegression(C=1.0, l1_ratio=0.0, solver="lbfgs", max_iter=5000)),
    ]
)

metricas_cinematica_lineal, predicciones_cinematica_lineal = evaluar_modelo_temporal(
    modelo_cinematica_lineal,
    "Control + cinemática lineal",
    df_entrenamiento_ml,
    pliegues_temporales_estandar,
)

comparacion_control_cinematica = metricas_control.merge(
    metricas_cinematica_lineal,
    on=["muestra", "pliegue", "anio_validacion"],
    suffixes=("_control", "_cinematica"),
    validate="one_to_one",
)

for metrica in columnas_metricas:
    comparacion_control_cinematica[f"delta_{metrica}"] = comparacion_control_cinematica[f"{metrica}_cinematica"] - comparacion_control_cinematica[f"{metrica}_control"]

filas_resumen_mejoras = []

for muestra, grupo in comparacion_control_cinematica.groupby("muestra"):
    fila = {"muestra": muestra}

    for metrica in columnas_metricas:
        diferencias = grupo[f"delta_{metrica}"]
        fila[f"delta_medio_{metrica}"] = diferencias.mean()
        fila[f"desviacion_delta_{metrica}"] = diferencias.std()
        fila[f"pliegues_con_mejora_{metrica}"] = (diferencias > 0).sum()

    filas_resumen_mejoras.append(fila)

resumen_mejoras_cinematica = pd.DataFrame(filas_resumen_mejoras)
columnas_comparacion_anual = ["muestra", "anio_validacion"] + [f"delta_{metrica}" for metrica in columnas_metricas]

display(metricas_cinematica_lineal.round(4))
display(comparacion_control_cinematica[columnas_comparacion_anual].round(4))
display(resumen_mejoras_cinematica.round(4))

if len(metricas_cinematica_lineal) != len(metricas_control):
    raise ValueError("El modelo cinemático y el modelo de control no produjeron el mismo número de evaluaciones.")

,modelo,muestra,pliegue,anio_validacion,trayectorias,brillantes,tenues,roc_auc,average_precision,exactitud,exactitud_balanceada,sensibilidad_tenues,especificidad_brillantes,verdaderos_brillantes,falsos_tenues,falsos_brillantes,verdaderos_tenues
0,Control + cinemática lineal,Estándar,1,2020,703,319,384,0.7709,0.8241,0.6942,0.6890,0.7448,0.6332,202,117,98,286
1,Control + cinemática lineal,Estricta,1,2020,116,56,60,0.8247,0.8064,0.6638,0.6714,0.4500,0.8929,50,6,33,27
2,Control + cinemática lineal,Estándar,2,2021,1319,437,882,0.8153,0.9000,0.7604,0.7337,0.8129,0.6545,286,151,165,717
3,Control + cinemática lineal,Estricta,2,2021,192,61,131,0.7600,0.8614,0.6667,0.7032,0.6031,0.8033,49,12,52,79
4,Control + cinemática lineal,Estándar,3,2022,2433,1405,1028,0.7088,0.6591,0.6449,0.6495,0.6790,0.6199,871,534,330,698
5,Control + cinemática lineal,Estricta,3,2022,1153,799,354,0.6653,0.5095,0.6357,0.6184,0.5734,0.6633,530,269,151,203
6,Control + cinemática lineal,Estándar,4,2023,2258,997,1261,0.7639,0.7919,0.6780,0.6915,0.5765,0.8064,804,193,534,727
7,Control + cinemática lineal,Estricta,4,2023,824,423,401,0.7879,0.7896,0.6626,0.6553,0.3815,0.9291,393,30,248,153
8,Control + cinemática lineal,Estándar,5,2024,1645,1011,634,0.8007,0.7373,0.7155,0.7244,0.7634,0.6855,693,318,150,484
9,Control + cinemática lineal,Estricta,5,2024,384,303,81,0.7326,0.4732,0.7656,0.6615,0.4815,0.8416,255,48,42,39


,muestra,anio_validacion,delta_roc_auc,delta_average_precision,delta_exactitud,delta_exactitud_balanceada,delta_sensibilidad_tenues,delta_especificidad_brillantes
0,Estándar,2020,0.0213,0.0246,0.0213,0.0224,0.0104,0.0345
1,Estricta,2020,0.0143,0.0197,0.0345,0.0321,0.1000,-0.0357
2,Estándar,2021,-0.0019,0.0096,-0.0008,0.0041,-0.0102,0.0183
3,Estricta,2021,-0.0086,-0.0004,-0.0260,-0.0235,-0.0305,-0.0164
4,Estándar,2022,0.0362,0.0391,0.0341,0.0271,-0.0185,0.0726
5,Estricta,2022,0.0529,0.0655,0.0642,0.0416,-0.0169,0.1001
6,Estándar,2023,0.0213,0.0178,0.0146,0.0160,0.0040,0.0281
7,Estricta,2023,0.0427,0.0523,0.0243,0.0246,0.0374,0.0118
8,Estándar,2024,0.0089,0.0135,0.0103,0.0087,0.0016,0.0158
9,Estricta,2024,0.0467,0.0546,-0.0078,0.0222,0.0741,-0.0297


,muestra,delta_medio_roc_auc,desviacion_delta_roc_auc,pliegues_con_mejora_roc_auc,delta_medio_average_precision,desviacion_delta_average_precision,pliegues_con_mejora_average_precision,delta_medio_exactitud,desviacion_delta_exactitud,pliegues_con_mejora_exactitud,delta_medio_exactitud_balanceada,desviacion_delta_exactitud_balanceada,pliegues_con_mejora_exactitud_balanceada,delta_medio_sensibilidad_tenues,desviacion_delta_sensibilidad_tenues,pliegues_con_mejora_sensibilidad_tenues,delta_medio_especificidad_brillantes,desviacion_delta_especificidad_brillantes,pliegues_con_mejora_especificidad_brillantes
0,Estricta,0.0296,0.0260,4,0.0384,0.0276,4,0.0178,0.0355,3,0.0194,0.0251,4,0.0328,0.0564,3,0.0060,0.0557,2
1,Estándar,0.0172,0.0144,4,0.0209,0.0116,5,0.0159,0.0130,4,0.0157,0.0095,5,-0.0025,0.0116,3,0.0339,0.0229,5


#### 12.2.2 Versión cuadrática

In [11]:
# Entrenar el modelo cinemático cuadrático y compararlo con los modelos anteriores
import sys
sys.path.insert(0, str(raiz_proyecto))
from src.geminidas_transformaciones import agregar_cuadrados_cinematicos

transformador_cinematica_cuadratica = Pipeline(
    [
        ("terminos_cinematicos", FunctionTransformer(agregar_cuadrados_cinematicos, validate=False)),
        ("escalamiento_cinematica", StandardScaler()),
    ]
)

preprocesador_control_cinematica_cuadratica = ColumnTransformer(
    [
        ("fase", transformador_fase, variables_fase),
        ("calidad", transformador_calidad, variables_calidad),
        ("cinematica", transformador_cinematica_cuadratica, variables_cinematica),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

modelo_cinematica_cuadratica = Pipeline(
    [
        ("preprocesamiento", preprocesador_control_cinematica_cuadratica),
        ("clasificador", LogisticRegression(C=1.0, l1_ratio=0.0, solver="lbfgs", max_iter=5000)),
    ]
)

metricas_cinematica_cuadratica, predicciones_cinematica_cuadratica = evaluar_modelo_temporal(
    modelo_cinematica_cuadratica,
    "Control + cinemática cuadrática",
    df_entrenamiento_ml,
    pliegues_temporales_estandar,
)

claves_comparacion = ["muestra", "pliegue", "anio_validacion"]
metricas_clave = ["roc_auc", "average_precision", "exactitud_balanceada"]

comparacion_tres_modelos = (
    metricas_control[claves_comparacion + metricas_clave]
    .rename(columns={metrica: f"{metrica}_control" for metrica in metricas_clave})
    .merge(
        metricas_cinematica_lineal[claves_comparacion + metricas_clave].rename(columns={metrica: f"{metrica}_lineal" for metrica in metricas_clave}),
        on=claves_comparacion,
        validate="one_to_one",
    )
    .merge(
        metricas_cinematica_cuadratica[claves_comparacion + metricas_clave].rename(columns={metrica: f"{metrica}_cuadratica" for metrica in metricas_clave}),
        on=claves_comparacion,
        validate="one_to_one",
    )
)

for metrica in metricas_clave:
    comparacion_tres_modelos[f"delta_{metrica}_cuadratica_vs_control"] = comparacion_tres_modelos[f"{metrica}_cuadratica"] - comparacion_tres_modelos[f"{metrica}_control"]
    comparacion_tres_modelos[f"delta_{metrica}_cuadratica_vs_lineal"] = comparacion_tres_modelos[f"{metrica}_cuadratica"] - comparacion_tres_modelos[f"{metrica}_lineal"]

filas_resumen_cuadratico = []

for muestra, grupo in comparacion_tres_modelos.groupby("muestra"):
    fila = {"muestra": muestra}

    for metrica in metricas_clave:
        diferencia_control = grupo[f"delta_{metrica}_cuadratica_vs_control"]
        diferencia_lineal = grupo[f"delta_{metrica}_cuadratica_vs_lineal"]
        fila[f"delta_medio_{metrica}_vs_control"] = diferencia_control.mean()
        fila[f"pliegues_mejora_{metrica}_vs_control"] = (diferencia_control > 0).sum()
        fila[f"delta_medio_{metrica}_vs_lineal"] = diferencia_lineal.mean()
        fila[f"pliegues_mejora_{metrica}_vs_lineal"] = (diferencia_lineal > 0).sum()

    filas_resumen_cuadratico.append(fila)

resumen_modelo_cuadratico = pd.DataFrame(filas_resumen_cuadratico)

display(metricas_cinematica_cuadratica.round(4))
display(comparacion_tres_modelos.round(4))
display(resumen_modelo_cuadratico.round(4))

if len(metricas_cinematica_cuadratica) != len(metricas_control):
    raise ValueError("El modelo cuadrático no produjo el número esperado de evaluaciones.")

,modelo,muestra,pliegue,anio_validacion,trayectorias,brillantes,tenues,roc_auc,average_precision,exactitud,exactitud_balanceada,sensibilidad_tenues,especificidad_brillantes,verdaderos_brillantes,falsos_tenues,falsos_brillantes,verdaderos_tenues
0,Control + cinemática cuadrática,Estándar,1,2020,703,319,384,0.7943,0.8372,0.7155,0.7083,0.7865,0.6301,201,118,82,302
1,Control + cinemática cuadrática,Estricta,1,2020,116,56,60,0.8283,0.8226,0.6638,0.6720,0.4333,0.9107,51,5,34,26
2,Control + cinemática cuadrática,Estándar,2,2021,1319,437,882,0.8277,0.9096,0.7665,0.7475,0.8039,0.6911,302,135,173,709
3,Control + cinemática cuadrática,Estricta,2,2021,192,61,131,0.7580,0.8685,0.6562,0.6912,0.5954,0.7869,48,13,53,78
4,Control + cinemática cuadrática,Estándar,3,2022,2433,1405,1028,0.7158,0.6728,0.6580,0.6625,0.6916,0.6335,890,515,317,711
5,Control + cinemática cuadrática,Estricta,3,2022,1153,799,354,0.6806,0.5354,0.6574,0.6474,0.6215,0.6733,538,261,134,220
6,Control + cinemática cuadrática,Estándar,4,2023,2258,997,1261,0.7740,0.7915,0.6842,0.6989,0.5734,0.8245,822,175,538,723
7,Control + cinemática cuadrática,Estricta,4,2023,824,423,401,0.7998,0.7940,0.6760,0.6688,0.4015,0.9362,396,27,240,161
8,Control + cinemática cuadrática,Estándar,5,2024,1645,1011,634,0.8105,0.7482,0.7277,0.7358,0.7713,0.7003,708,303,145,489
9,Control + cinemática cuadrática,Estricta,5,2024,384,303,81,0.7508,0.4997,0.7630,0.6599,0.4815,0.8383,254,49,42,39


,muestra,pliegue,anio_validacion,roc_auc_control,average_precision_control,exactitud_balanceada_control,roc_auc_lineal,average_precision_lineal,exactitud_balanceada_lineal,roc_auc_cuadratica,average_precision_cuadratica,exactitud_balanceada_cuadratica,delta_roc_auc_cuadratica_vs_control,delta_roc_auc_cuadratica_vs_lineal,delta_average_precision_cuadratica_vs_control,delta_average_precision_cuadratica_vs_lineal,delta_exactitud_balanceada_cuadratica_vs_control,delta_exactitud_balanceada_cuadratica_vs_lineal
0,Estándar,1,2020,0.7496,0.7995,0.6666,0.7709,0.8241,0.6890,0.7943,0.8372,0.7083,0.0447,0.0234,0.0377,0.0131,0.0417,0.0193
1,Estricta,1,2020,0.8104,0.7867,0.6393,0.8247,0.8064,0.6714,0.8283,0.8226,0.6720,0.0179,0.0036,0.0358,0.0161,0.0327,0.0006
2,Estándar,2,2021,0.8172,0.8904,0.7296,0.8153,0.9000,0.7337,0.8277,0.9096,0.7475,0.0105,0.0124,0.0192,0.0096,0.0178,0.0138
3,Estricta,2,2021,0.7686,0.8618,0.7266,0.7600,0.8614,0.7032,0.7580,0.8685,0.6912,-0.0106,-0.0020,0.0067,0.0071,-0.0355,-0.0120
4,Estándar,3,2022,0.6726,0.6200,0.6224,0.7088,0.6591,0.6495,0.7158,0.6728,0.6625,0.0432,0.0070,0.0528,0.0137,0.0401,0.0131
5,Estricta,3,2022,0.6124,0.4440,0.5768,0.6653,0.5095,0.6184,0.6806,0.5354,0.6474,0.0682,0.0153,0.0914,0.0258,0.0706,0.0290
6,Estándar,4,2023,0.7426,0.7741,0.6754,0.7639,0.7919,0.6915,0.7740,0.7915,0.6989,0.0314,0.0101,0.0174,-0.0004,0.0235,0.0074
7,Estricta,4,2023,0.7452,0.7373,0.6307,0.7879,0.7896,0.6553,0.7998,0.7940,0.6688,0.0546,0.0119,0.0567,0.0044,0.0381,0.0135
8,Estándar,5,2024,0.7918,0.7238,0.7157,0.8007,0.7373,0.7244,0.8105,0.7482,0.7358,0.0187,0.0098,0.0244,0.0109,0.0201,0.0114
9,Estricta,5,2024,0.6859,0.4186,0.6393,0.7326,0.4732,0.6615,0.7508,0.4997,0.6599,0.0649,0.0183,0.0811,0.0265,0.0205,-0.0017


,muestra,delta_medio_roc_auc_vs_control,pliegues_mejora_roc_auc_vs_control,delta_medio_roc_auc_vs_lineal,pliegues_mejora_roc_auc_vs_lineal,delta_medio_average_precision_vs_control,pliegues_mejora_average_precision_vs_control,delta_medio_average_precision_vs_lineal,pliegues_mejora_average_precision_vs_lineal,delta_medio_exactitud_balanceada_vs_control,pliegues_mejora_exactitud_balanceada_vs_control,delta_medio_exactitud_balanceada_vs_lineal,pliegues_mejora_exactitud_balanceada_vs_lineal
0,Estricta,0.0390,4,0.0094,4,0.0544,5,0.0160,5,0.0253,4,0.0059,3
1,Estándar,0.0297,5,0.0125,5,0.0303,5,0.0094,4,0.0286,5,0.0130,5


### 12.3 Modelo PCA lineal

In [12]:
from sklearn.decomposition import PCA

resultados_pca_temporal = []

for anio in [2020, 2021, 2022, 2023, 2024]:

    # Meteoros disponibles antes del año de validación
    mascara_ajuste = (
        df_entrenamiento["usar_pca"]
        & (df_entrenamiento["source_year"] < anio)
    )

    X_ajuste = df_entrenamiento.loc[
        mascara_ajuste, variables_cinematica
    ]

    # Meteoros brillantes y tenues del año que vamos a validar
    X_validacion = df_entrenamiento_ml.loc[
        df_entrenamiento_ml["source_year"] == anio,
        variables_cinematica
    ]

    # Ajustar los ejes solamente con los años anteriores
    pca_temporal = PCA(n_components=3)
    pca_temporal.fit(X_ajuste)

    # Aplicar los ejes aprendidos al año de validación
    Z_validacion = pca_temporal.transform(X_validacion)

    resultados_pca_temporal.append({
        "anio_validacion": anio,
        "n_ajuste_pca": len(X_ajuste),
        "n_validacion_ml": len(X_validacion),
        "PC1_%": 100 * pca_temporal.explained_variance_ratio_[0],
        "PC2_%": 100 * pca_temporal.explained_variance_ratio_[1],
        "PC3_%": 100 * pca_temporal.explained_variance_ratio_[2],
        "proyeccion_valida": np.isfinite(Z_validacion).all()
    })

tabla_pca_temporal = pd.DataFrame(resultados_pca_temporal)

display(tabla_pca_temporal.round(2))

assert tabla_pca_temporal["proyeccion_valida"].all()

,anio_validacion,n_ajuste_pca,n_validacion_ml,PC1_%,PC2_%,PC3_%,proyeccion_valida
0,2020,430,703,74.32,14.50,11.19,True
1,2021,1432,1319,79.49,11.47,9.04,True
2,2022,3336,2433,80.87,11.13,8.00,True
3,2023,7048,2258,80.47,11.66,7.87,True
4,2024,10445,1645,79.84,12.06,8.10,True


12.3.1 PCA lineal con dos componentes

In [13]:
from sklearn.decomposition import PCA

def preparar_pca_pliegue(X_entrenamiento, X_validacion, anio_validacion):

    # El PCA utiliza todos los meteoros elegibles de años anteriores
    mascara_pca = (
        df_entrenamiento["usar_pca"]
        & (df_entrenamiento["source_year"] < anio_validacion)
    )

    datos_pca = df_entrenamiento.loc[mascara_pca, variables_cinematica]

    pca_pliegue = PCA(n_components=3)
    pca_pliegue.fit(datos_pca)

    # Proyectar entrenamiento y validación con los mismos ejes
    Z_entrenamiento = pca_pliegue.transform(
        X_entrenamiento[variables_cinematica]
    )

    Z_validacion = pca_pliegue.transform(
        X_validacion[variables_cinematica]
    )

    for i in range(3):
        columna = f"PC{i+1}"
        X_entrenamiento[columna] = Z_entrenamiento[:, i]
        X_validacion[columna] = Z_validacion[:, i]

    return X_entrenamiento, X_validacion


# Fase y calidad se procesan igual que en los modelos de Alex
preprocesador_pca_2 = ColumnTransformer(
    [
        ("fase", clone(transformador_fase), variables_fase),
        ("calidad", clone(transformador_calidad), variables_calidad),
        ("pca", StandardScaler(), ["PC1", "PC2"]),
    ],
    remainder="drop",
)


modelo_pca_2 = Pipeline(
    [
        ("preprocesamiento", preprocesador_pca_2),
        (
            "clasificador",
            LogisticRegression(
                C=1.0,
                l1_ratio=0.0,
                solver="lbfgs",
                max_iter=5000,
            ),
        ),
    ]
)


metricas_pca_2, predicciones_pca_2 = evaluar_modelo_temporal(
    modelo_pca_2,
    "Control + PCA lineal (2 PC)",
    df_entrenamiento_ml,
    pliegues_temporales_estandar,
    preparar_pliegue=preparar_pca_pliegue,
)

display(metricas_pca_2.round(4))

,modelo,muestra,pliegue,anio_validacion,trayectorias,brillantes,tenues,roc_auc,average_precision,exactitud,exactitud_balanceada,sensibilidad_tenues,especificidad_brillantes,verdaderos_brillantes,falsos_tenues,falsos_brillantes,verdaderos_tenues
0,Control + PCA lineal (2 PC),Estándar,1,2020,703,319,384,0.7709,0.8232,0.7027,0.6971,0.7578,0.6364,203,116,93,291
1,Control + PCA lineal (2 PC),Estricta,1,2020,116,56,60,0.8280,0.8074,0.6552,0.6631,0.4333,0.8929,50,6,34,26
2,Control + PCA lineal (2 PC),Estándar,2,2021,1319,437,882,0.8139,0.8990,0.7582,0.7320,0.8095,0.6545,286,151,168,714
3,Control + PCA lineal (2 PC),Estricta,2,2021,192,61,131,0.7560,0.8608,0.6719,0.7157,0.5954,0.8361,51,10,53,78
4,Control + PCA lineal (2 PC),Estándar,3,2022,2433,1405,1028,0.7100,0.6591,0.6473,0.6522,0.6839,0.6206,872,533,325,703
5,Control + PCA lineal (2 PC),Estricta,3,2022,1153,799,354,0.6691,0.5110,0.6375,0.6204,0.5763,0.6646,531,268,150,204
6,Control + PCA lineal (2 PC),Estándar,4,2023,2258,997,1261,0.7607,0.7889,0.6780,0.6905,0.5837,0.7974,795,202,525,736
7,Control + PCA lineal (2 PC),Estricta,4,2023,824,423,401,0.7876,0.7879,0.6675,0.6603,0.3915,0.9291,393,30,244,157
8,Control + PCA lineal (2 PC),Estándar,5,2024,1645,1011,634,0.7986,0.7338,0.7100,0.7206,0.7666,0.6746,682,329,148,486
9,Control + PCA lineal (2 PC),Estricta,5,2024,384,303,81,0.7265,0.4620,0.7578,0.6566,0.4815,0.8317,252,51,42,39


In [14]:
comparacion_pca_2 = (
    metricas_pca_2.loc[
        metricas_pca_2["muestra"] == "Estándar",
        ["anio_validacion", "roc_auc", "average_precision"]
    ]
    .merge(
        metricas_control.loc[
            metricas_control["muestra"] == "Estándar",
            ["anio_validacion", "roc_auc", "average_precision"]
        ],
        on="anio_validacion",
        suffixes=("_pca", "_control"),
        validate="one_to_one",
    )
)

comparacion_pca_2["mejora_roc_auc"] = (
    comparacion_pca_2["roc_auc_pca"]
    - comparacion_pca_2["roc_auc_control"]
)

display(comparacion_pca_2.round(4))

print(
    "Mejora media de ROC-AUC:",
    round(comparacion_pca_2["mejora_roc_auc"].mean(), 4)
)

,anio_validacion,roc_auc_pca,average_precision_pca,roc_auc_control,average_precision_control,mejora_roc_auc
0,2020,0.7709,0.8232,0.7496,0.7995,0.0213
1,2021,0.8139,0.8990,0.8172,0.8904,-0.0033
2,2022,0.7100,0.6591,0.6726,0.6200,0.0374
3,2023,0.7607,0.7889,0.7426,0.7741,0.0181
4,2024,0.7986,0.7338,0.7918,0.7238,0.0068


Mejora media de ROC-AUC: 0.0161


#### 12.3.2 PCA lineal con tres componentes

In [15]:
# Modelo lineal con las tres componentes principales

preprocesador_pca_3 = ColumnTransformer(
    [
        ("fase", clone(transformador_fase), variables_fase),
        ("calidad", clone(transformador_calidad), variables_calidad),
        ("pca", StandardScaler(), ["PC1", "PC2", "PC3"]),
    ],
    remainder="drop",
)

modelo_pca_3 = Pipeline(
    [
        ("preprocesamiento", preprocesador_pca_3),
        (
            "clasificador",
            LogisticRegression(
                C=1.0,
                l1_ratio=0.0,
                solver="lbfgs",
                max_iter=5000,
            ),
        ),
    ]
)

metricas_pca_3, predicciones_pca_3 = evaluar_modelo_temporal(
    modelo_pca_3,
    "Control + PCA lineal (3 PC)",
    df_entrenamiento_ml,
    pliegues_temporales_estandar,
    preparar_pliegue=preparar_pca_pliegue,
)

display(
    metricas_pca_3[
        ["muestra", "anio_validacion", "roc_auc", "average_precision",
         "exactitud_balanceada"]
    ].round(4)
)

,muestra,anio_validacion,roc_auc,average_precision,exactitud_balanceada
0,Estándar,2020,0.7708,0.8241,0.6890
1,Estricta,2020,0.8241,0.8062,0.6714
2,Estándar,2021,0.8153,0.9000,0.7337
3,Estricta,2021,0.7604,0.8616,0.7032
4,Estándar,2022,0.7088,0.6591,0.6495
5,Estricta,2022,0.6653,0.5094,0.6184
6,Estándar,2023,0.7639,0.7919,0.6919
7,Estricta,2023,0.7879,0.7896,0.6553
8,Estándar,2024,0.8007,0.7374,0.7244
9,Estricta,2024,0.7326,0.4732,0.6615


In [16]:
# Comparar los modelos sobre los mismos años

modelos_lineales = {
    "Control": metricas_control,
    "Local_3": metricas_cinematica_lineal,
    "PCA_2": metricas_pca_2,
    "PCA_3": metricas_pca_3,
}

tabla_lineales = None

for nombre, metricas in modelos_lineales.items():
    datos = metricas.loc[
        metricas["muestra"] == "Estándar",
        ["anio_validacion", "roc_auc"]
    ].rename(columns={"roc_auc": nombre})

    if tabla_lineales is None:
        tabla_lineales = datos
    else:
        tabla_lineales = tabla_lineales.merge(
            datos,
            on="anio_validacion",
            validate="one_to_one"
        )

tabla_lineales["PCA_2_vs_control"] = (
    tabla_lineales["PCA_2"] - tabla_lineales["Control"]
)

tabla_lineales["PCA_3_vs_control"] = (
    tabla_lineales["PCA_3"] - tabla_lineales["Control"]
)

tabla_lineales["PCA_3_vs_PCA_2"] = (
    tabla_lineales["PCA_3"] - tabla_lineales["PCA_2"]
)

display(tabla_lineales.round(4))

print("\nPromedios por modelo:")
display(
    tabla_lineales[
        ["Control", "Local_3", "PCA_2", "PCA_3"]
    ].mean().round(4)
)

,anio_validacion,Control,Local_3,PCA_2,PCA_3,PCA_2_vs_control,PCA_3_vs_control,PCA_3_vs_PCA_2
0,2020,0.7496,0.7709,0.7709,0.7708,0.0213,0.0212,-0.0001
1,2021,0.8172,0.8153,0.8139,0.8153,-0.0033,-0.0019,0.0014
2,2022,0.6726,0.7088,0.7100,0.7088,0.0374,0.0362,-0.0012
3,2023,0.7426,0.7639,0.7607,0.7639,0.0181,0.0213,0.0032
4,2024,0.7918,0.8007,0.7986,0.8007,0.0068,0.0089,0.0021



Promedios por modelo:


Control    0.7548
Local_3    0.7719
PCA_2      0.7708
PCA_3      0.7719
dtype: float64

### 12.4 PCA cuadrático

In [17]:
# PCA cuadrático diagonal con dos componentes

modelo_pca_2_cuadratico = clone(modelo_pca_2)

modelo_pca_2_cuadratico.set_params(
    preprocesamiento__pca=Pipeline([
        (
            "cuadrados",
            FunctionTransformer(
                agregar_cuadrados_cinematicos,
                validate=False
            )
        ),
        ("escalamiento", StandardScaler())
    ])
)

metricas_pca_2_cuadratico, predicciones_pca_2_cuadratico = (
    evaluar_modelo_temporal(
        modelo_pca_2_cuadratico,
        "Control + PCA cuadratico diagonal (2 PC)",
        df_entrenamiento_ml,
        pliegues_temporales_estandar,
        preparar_pliegue=preparar_pca_pliegue,
    )
)

# Comparar con el PCA lineal de dos componentes
comparacion_pca_cuadratico = (
    metricas_pca_2_cuadratico.loc[
        metricas_pca_2_cuadratico["muestra"] == "Estándar",
        ["anio_validacion", "roc_auc"]
    ]
    .rename(columns={"roc_auc": "PCA_2_cuadratico"})
    .merge(
        metricas_pca_2.loc[
            metricas_pca_2["muestra"] == "Estándar",
            ["anio_validacion", "roc_auc"]
        ].rename(columns={"roc_auc": "PCA_2_lineal"}),
        on="anio_validacion",
        validate="one_to_one"
    )
)

comparacion_pca_cuadratico["mejora_cuadratico"] = (
    comparacion_pca_cuadratico["PCA_2_cuadratico"]
    - comparacion_pca_cuadratico["PCA_2_lineal"]
)

display(comparacion_pca_cuadratico.round(4))

print(
    "Mejora media:",
    comparacion_pca_cuadratico["mejora_cuadratico"].mean().round(4)
)

print(
    "Años con mejora:",
    (comparacion_pca_cuadratico["mejora_cuadratico"] > 0).sum(),
    "de 5"
)

,anio_validacion,PCA_2_cuadratico,PCA_2_lineal,mejora_cuadratico
0,2020,0.7913,0.7709,0.0204
1,2021,0.8244,0.8139,0.0105
2,2022,0.7149,0.7100,0.0049
3,2023,0.7701,0.7607,0.0093
4,2024,0.8068,0.7986,0.0082


Mejora media: 0.0107
Años con mejora: 5 de 5


In [18]:
# PCA cuadrático con tres componentes

preprocesador_pca_3_cuadratico = ColumnTransformer(
    [
        ("fase", clone(transformador_fase), variables_fase),
        ("calidad", clone(transformador_calidad), variables_calidad),
        (
            "pca",
            Pipeline([
                (
                    "cuadrados",
                    FunctionTransformer(
                        agregar_cuadrados_cinematicos,
                        validate=False
                    )
                ),
                ("escalamiento", StandardScaler())
            ]),
            ["PC1", "PC2", "PC3"]
        ),
    ],
    remainder="drop",
)

modelo_pca_3_cuadratico = Pipeline([
    ("preprocesamiento", preprocesador_pca_3_cuadratico),
    ("clasificador", clone(modelo_pca_3.named_steps["clasificador"]))
])

metricas_pca_3_cuadratico, predicciones_pca_3_cuadratico = (
    evaluar_modelo_temporal(
        modelo_pca_3_cuadratico,
        "Control + PCA cuadratico diagonal (3 PC)",
        df_entrenamiento_ml,
        pliegues_temporales_estandar,
        preparar_pliegue=preparar_pca_pliegue,
    )
)

comparacion_cuadraticos = (
    metricas_pca_2_cuadratico.query("muestra == 'Estándar'")
    [["anio_validacion", "roc_auc"]]
    .rename(columns={"roc_auc": "PCA_2"})
    .merge(
        metricas_pca_3_cuadratico.query("muestra == 'Estándar'")
        [["anio_validacion", "roc_auc"]]
        .rename(columns={"roc_auc": "PCA_3"}),
        on="anio_validacion",
        validate="one_to_one"
    )
)

comparacion_cuadraticos["diferencia"] = (
    comparacion_cuadraticos["PCA_3"]
    - comparacion_cuadraticos["PCA_2"]
)

display(comparacion_cuadraticos.round(4))
print("Diferencia media:", comparacion_cuadraticos["diferencia"].mean().round(4))

,anio_validacion,PCA_2,PCA_3,diferencia
0,2020,0.7913,0.7871,-0.0042
1,2021,0.8244,0.8289,0.0045
2,2022,0.7149,0.7175,0.0027
3,2023,0.7701,0.7738,0.0037
4,2024,0.8068,0.8100,0.0032


Diferencia media: 0.002


### 12.5 Comparación de modelso

In [19]:
# Comparar los siete modelos de clasificación

modelos = {
    "Control": metricas_control,
    "Local lineal": metricas_cinematica_lineal,
    "Local cuadrático": metricas_cinematica_cuadratica,
    "PCA 2 lineal": metricas_pca_2,
    "PCA 3 lineal": metricas_pca_3,
    "PCA 2 cuadrático": metricas_pca_2_cuadratico,
    "PCA 3 cuadrático": metricas_pca_3_cuadratico,
}

columnas = [
    "muestra", "anio_validacion", "trayectorias",
    "roc_auc", "average_precision", "exactitud_balanceada"
]

tabla_modelos = pd.concat(
    [
        metricas[columnas].assign(modelo=nombre)
        for nombre, metricas in modelos.items()
    ],
    ignore_index=True
)

# Comprobar que los modelos se comparan en los mismos años y tamaños
claves = ["muestra", "anio_validacion"]

assert not tabla_modelos.duplicated(claves + ["modelo"]).any()
assert (tabla_modelos.groupby(claves)["trayectorias"].nunique() == 1).all()
assert (tabla_modelos.groupby(claves)["modelo"].nunique() == 7).all()

# Añadir las métricas del control de cada año
control = tabla_modelos.loc[
    tabla_modelos["modelo"] == "Control",
    claves + ["roc_auc", "average_precision", "exactitud_balanceada"]
].rename(columns={
    "roc_auc": "auc_control",
    "average_precision": "ap_control",
    "exactitud_balanceada": "balanceada_control"
})

tabla_modelos = tabla_modelos.merge(
    control, on=claves, validate="many_to_one"
)

tabla_modelos["delta_auc"] = (
    tabla_modelos["roc_auc"] - tabla_modelos["auc_control"]
)

tabla_modelos["delta_ap"] = (
    tabla_modelos["average_precision"] - tabla_modelos["ap_control"]
)

tabla_modelos["delta_balanceada"] = (
    tabla_modelos["exactitud_balanceada"]
    - tabla_modelos["balanceada_control"]
)

# Desempeño promedio y mejoras respecto al control
resumen_modelos = tabla_modelos.groupby(
    ["muestra", "modelo"], as_index=False
).agg(
    auc_medio=("roc_auc", "mean"),
    ap_medio=("average_precision", "mean"),
    balanceada_media=("exactitud_balanceada", "mean"),
    mejora_auc=("delta_auc", "mean"),
    mejora_ap=("delta_ap", "mean"),
    anios_mejora=("delta_auc", lambda x: (x > 0).sum())
)

display(
    resumen_modelos.sort_values(
        ["muestra", "mejora_auc"],
        ascending=[True, False]
    ).round(4)
)

# Comparación anual de la muestra estándar
tabla_anual = tabla_modelos.loc[
    tabla_modelos["muestra"] == "Estándar"
].pivot(
    index="anio_validacion",
    columns="modelo",
    values="roc_auc"
)

display(tabla_anual.round(4))

,muestra,modelo,auc_medio,ap_medio,balanceada_media,mejora_auc,mejora_ap,anios_mejora
1,Estricta,Local cuadrático,0.7635,0.7040,0.6679,0.0390,0.0544,4
5,Estricta,PCA 3 cuadrático,0.7630,0.7043,0.6676,0.0385,0.0546,4
3,Estricta,PCA 2 cuadrático,0.7595,0.6994,0.6613,0.0350,0.0497,4
2,Estricta,Local lineal,0.7541,0.6880,0.6620,0.0296,0.0384,4
6,Estricta,PCA 3 lineal,0.7541,0.6880,0.6620,0.0295,0.0383,4
4,Estricta,PCA 2 lineal,0.7534,0.6858,0.6632,0.0289,0.0361,4
0,Estricta,Control,0.7245,0.6497,0.6426,0.0000,0.0000,0
8,Estándar,Local cuadrático,0.7845,0.7919,0.7106,0.0297,0.0303,5
12,Estándar,PCA 3 cuadrático,0.7835,0.7913,0.7087,0.0287,0.0297,5
10,Estándar,PCA 2 cuadrático,0.7815,0.7889,0.7069,0.0267,0.0273,5


modelo,Control,Local cuadrático,Local lineal,PCA 2 cuadrático,PCA 2 lineal,PCA 3 cuadrático,PCA 3 lineal
anio_validacion,,,,,,,
2020,0.7496,0.7943,0.7709,0.7913,0.7709,0.7871,0.7708
2021,0.8172,0.8277,0.8153,0.8244,0.8139,0.8289,0.8153
2022,0.6726,0.7158,0.7088,0.7149,0.7100,0.7175,0.7088
2023,0.7426,0.7740,0.7639,0.7701,0.7607,0.7738,0.7639
2024,0.7918,0.8105,0.8007,0.8068,0.7986,0.8100,0.8007


### 12.6 Revisión de las componentes cinemáticas por fase solar

In [20]:
# Resumen de las componentes residuales por intervalos de fase solar

base_fase = df_entrenamiento.loc[
    df_entrenamiento["usar_pca"],
    ["delta_sol_lon_deg"] + variables_cinematica
].copy()

limites_fase = np.arange(-5, 2.5, 0.5)

base_fase["intervalo_fase"] = pd.cut(
    base_fase["delta_sol_lon_deg"],
    bins=limites_fase,
    right=False
)

fuera_ventana = base_fase["intervalo_fase"].isna().sum()
print("Trayectorias fuera de la ventana de fase:", fuera_ventana)

resumen_fase = base_fase.groupby(
    "intervalo_fase", observed=True
).agg(
    n=("delta_sol_lon_deg", "size"),
    media_paralela=(variables_cinematica[0], "mean"),
    sd_paralela=(variables_cinematica[0], "std"),
    media_lambda=(variables_cinematica[1], "mean"),
    sd_lambda=(variables_cinematica[1], "std"),
    media_beta=(variables_cinematica[2], "mean"),
    sd_beta=(variables_cinematica[2], "std")
)

display(resumen_fase.round(4))

Trayectorias fuera de la ventana de fase: 0


,n,media_paralela,sd_paralela,media_lambda,sd_lambda,media_beta,sd_beta
intervalo_fase,,,,,,,
"[-5.0, -4.5)",124,0.0045,0.7919,-0.0310,0.3290,-0.0078,0.3433
"[-4.5, -4.0)",145,0.0883,0.7928,-0.0078,0.3362,0.0336,0.3416
"[-4.0, -3.5)",166,0.0946,0.7277,0.0115,0.3322,-0.0347,0.3437
"[-3.5, -3.0)",234,0.0166,0.8465,-0.0426,0.3441,-0.0096,0.3275
"[-3.0, -2.5)",231,-0.0429,0.7553,-0.0171,0.3019,0.0087,0.3257
"[-2.5, -2.0)",382,-0.0249,0.7106,-0.0080,0.2768,0.0077,0.2634
"[-2.0, -1.5)",679,-0.0000,0.7018,-0.0147,0.2816,0.0005,0.2409
"[-1.5, -1.0)",1030,-0.0209,0.6811,-0.0006,0.2440,0.0075,0.2235
"[-1.0, -0.5)",2215,-0.0293,0.5867,0.0147,0.2329,-0.0054,0.2176


#### Dispersión por brillo dentro de intervalos de fase

In [21]:
# Comparar la dispersión cinemática por brillo y fase solar

base_fase_brillo = df_entrenamiento.loc[
    df_entrenamiento["usar_modelo_ml"],
    ["delta_sol_lon_deg", "grupo_brillo_ml"] + variables_cinematica
].copy()

base_fase_brillo["intervalo_fase"] = pd.cut(
    base_fase_brillo["delta_sol_lon_deg"],
    bins=np.arange(-5, 2.5, 0.5),
    right=False
)

# Calcular el tamaño y la dispersión de cada grupo
tabla_grupos = (
    base_fase_brillo
    .groupby(["intervalo_fase", "grupo_brillo_ml"], observed=True)
    .agg(
        n=("grupo_brillo_ml", "size"),
        sd_paralela=(variables_cinematica[0], "std"),
        sd_lambda=(variables_cinematica[1], "std"),
        sd_beta=(variables_cinematica[2], "std")
    )
    .unstack("grupo_brillo_ml")
)

# Comparar la desviación estándar de tenues contra brillantes
tabla_dispersion_fase = pd.DataFrame(index=tabla_grupos.index)

for grupo in ["brillante", "tenue"]:
    tabla_dispersion_fase[f"n_{grupo}"] = (
        tabla_grupos[("n", grupo)].fillna(0).astype(int)
    )

for eje in ["paralela", "lambda", "beta"]:
    tabla_dispersion_fase[f"razon_{eje}"] = (
        tabla_grupos[(f"sd_{eje}", "tenue")]
        / tabla_grupos[(f"sd_{eje}", "brillante")]
    )

# Evitar interpretar intervalos con muy pocas observaciones
tabla_dispersion_fase["n_suficiente"] = (
    (tabla_dispersion_fase["n_brillante"] >= 50)
    & (tabla_dispersion_fase["n_tenue"] >= 50)
)

display(tabla_dispersion_fase.round(3))

print(
    "Intervalos con al menos 50 meteoros por grupo:",
    tabla_dispersion_fase["n_suficiente"].sum(),
    "de",
    len(tabla_dispersion_fase)
)

,n_brillante,n_tenue,razon_paralela,razon_lambda,razon_beta,n_suficiente
intervalo_fase,,,,,,
"[-5.0, -4.5)",35,48,1.601,1.206,0.944,False
"[-4.5, -4.0)",33,54,1.329,1.063,0.843,False
"[-4.0, -3.5)",55,63,1.214,1.069,1.003,True
"[-3.5, -3.0)",62,106,1.603,1.047,0.775,True
"[-3.0, -2.5)",56,103,1.088,1.065,0.727,True
"[-2.5, -2.0)",75,189,1.349,0.888,0.756,True
"[-2.0, -1.5)",172,285,1.431,0.888,0.909,True
"[-1.5, -1.0)",224,474,1.649,0.985,0.861,True
"[-1.0, -0.5)",586,890,1.417,1.054,0.975,True


Intervalos con al menos 50 meteoros por grupo: 11 de 14


#### Comparaciones de calidad observacional

In [22]:
# Calidad observacional de brillantes y tenues

calidad_por_brillo = (
    df_entrenamiento_ml
    .groupby("grupo_brillo_ml", observed=True)[variables_calidad]
    .median()
    .T
)

display(calidad_por_brillo.round(4))

# Cuántos meteoros de cada grupo conserva la muestra estricta
retencion_calidad = (
    df_entrenamiento_ml
    .groupby("grupo_brillo_ml", observed=True)["pertenece_muestra_estricta"]
    .agg(n="size", estrictas="sum")
)

retencion_calidad["porcentaje_estricto"] = (
    100 * retencion_calidad["estrictas"] / retencion_calidad["n"]
)

display(retencion_calidad.round(2))

grupo_brillo_ml,brillante,tenue
Qc (deg),68.29,63.63
MedianFitErr (arcsec),46.53,39.785
numero_estaciones,3.0,2.0
sigma_radiante_aprox,0.148215,0.174549
Vgeo_sigma_modelo,0.0413,0.08775


,n,estrictas,porcentaje_estricto
grupo_brillo_ml,,,
brillante,4300,1681,39.09
tenue,4332,1046,24.15


In [23]:
# Comparación por fase utilizando solamente la muestra estricta

base_estricta_fase = df_entrenamiento_ml_estricto.copy()

base_estricta_fase["intervalo_fase"] = pd.cut(
    base_estricta_fase["delta_sol_lon_deg"],
    bins=np.arange(-5, 2.5, 0.5),
    right=False
)

resumen_estricto = (
    base_estricta_fase
    .groupby(
        ["intervalo_fase", "grupo_brillo_ml"],
        observed=True
    )[variables_cinematica[0]]
    .agg(n="size", sd="std")
    .unstack("grupo_brillo_ml")
)

comparacion_calidad_fase = (
    tabla_dispersion_fase[
        ["n_brillante", "n_tenue", "razon_paralela"]
    ]
    .rename(columns={"razon_paralela": "razon_estandar"})
    .copy()
)

for grupo in ["brillante", "tenue"]:
    comparacion_calidad_fase[f"n_estricto_{grupo}"] = (
        resumen_estricto[("n", grupo)]
        .reindex(comparacion_calidad_fase.index)
        .fillna(0)
        .astype(int)
    )

comparacion_calidad_fase["razon_estricta"] = (
    resumen_estricto[("sd", "tenue")]
    / resumen_estricto[("sd", "brillante")]
)

# Un mínimo exploratorio de 30 observaciones por clase
comparacion_calidad_fase["revisar"] = (
    (comparacion_calidad_fase["n_estricto_brillante"] >= 30)
    & (comparacion_calidad_fase["n_estricto_tenue"] >= 30)
)

display(comparacion_calidad_fase.round(3))

intervalos_revisables = comparacion_calidad_fase.loc[
    comparacion_calidad_fase["revisar"]
]

print("Intervalos revisables:", len(intervalos_revisables))

if len(intervalos_revisables) > 0:
    print(
        "Razón estricta > 1:",
        (intervalos_revisables["razon_estricta"] > 1).sum(),
        "de",
        len(intervalos_revisables)
    )

,n_brillante,n_tenue,razon_estandar,n_estricto_brillante,n_estricto_tenue,razon_estricta,revisar
intervalo_fase,,,,,,,
"[-5.0, -4.5)",35,48,1.601,18,22,1.575,False
"[-4.5, -4.0)",33,54,1.329,22,15,1.318,False
"[-4.0, -3.5)",55,63,1.214,24,19,1.218,False
"[-3.5, -3.0)",62,106,1.603,18,29,1.755,False
"[-3.0, -2.5)",56,103,1.088,22,24,1.405,False
"[-2.5, -2.0)",75,189,1.349,24,43,1.270,False
"[-2.0, -1.5)",172,285,1.431,63,61,1.595,True
"[-1.5, -1.0)",224,474,1.649,68,87,1.411,True
"[-1.0, -0.5)",586,890,1.417,221,234,1.534,True


Intervalos revisables: 6
Razón estricta > 1: 6 de 6


### 12.5.2 Comprobaciones de los modelos

In [24]:
# Comprobar que todos los modelos evaluaron las mismas trayectorias

predicciones_modelos = {
    "Control": predicciones_control,
    "Local lineal": predicciones_cinematica_lineal,
    "Local cuadrático": predicciones_cinematica_cuadratica,
    "PCA 2 lineal": predicciones_pca_2,
    "PCA 3 lineal": predicciones_pca_3,
    "PCA 2 cuadrático": predicciones_pca_2_cuadratico,
    "PCA 3 cuadrático": predicciones_pca_3_cuadratico,
}

columnas_revision = [
    "id_fila_origen",
    "anio_validacion",
    "objetivo_real",
    "pertenece_muestra_estricta"
]

referencia = (
    predicciones_control[columnas_revision]
    .sort_values(["anio_validacion", "id_fila_origen"])
    .reset_index(drop=True)
)

filas_revision = []

for nombre, predicciones in predicciones_modelos.items():

    datos = (
        predicciones[columnas_revision]
        .sort_values(["anio_validacion", "id_fila_origen"])
        .reset_index(drop=True)
    )

    mismas_filas = datos.equals(referencia)

    probabilidades = predicciones["probabilidad_tenue"]
    probabilidades_validas = (
        probabilidades.between(0, 1).all()
        and np.isfinite(probabilidades).all()
    )

    ids_unicos = predicciones["id_fila_origen"].is_unique

    filas_revision.append({
        "modelo": nombre,
        "mismas_filas": mismas_filas,
        "ids_unicos": ids_unicos,
        "probabilidades_validas": probabilidades_validas
    })

auditoria_predicciones = pd.DataFrame(filas_revision)

display(auditoria_predicciones)

assert auditoria_predicciones.iloc[:, 1:].to_numpy().all()

,modelo,mismas_filas,ids_unicos,probabilidades_validas
0,Control,True,True,True
1,Local lineal,True,True,True
2,Local cuadrático,True,True,True
3,PCA 2 lineal,True,True,True
4,PCA 3 lineal,True,True,True
5,PCA 2 cuadrático,True,True,True
6,PCA 3 cuadrático,True,True,True


In [25]:
# Comprobar convergencia en los cinco pliegues temporales

import warnings
from sklearn.exceptions import ConvergenceWarning

modelos_auditar = {
    "Control": (modelo_control, None),
    "Local lineal": (modelo_cinematica_lineal, None),
    "Local cuadrático": (modelo_cinematica_cuadratica, None),
    "PCA 2 lineal": (modelo_pca_2, preparar_pca_pliegue),
    "PCA 3 lineal": (modelo_pca_3, preparar_pca_pliegue),
    "PCA 2 cuadrático": (
        modelo_pca_2_cuadratico, preparar_pca_pliegue
    ),
    "PCA 3 cuadrático": (
        modelo_pca_3_cuadratico, preparar_pca_pliegue
    ),
}

revision_convergencia = []

for nombre, (modelo, preparar) in modelos_auditar.items():

    for indices_train, indices_val in pliegues_temporales_estandar:

        X_train = df_entrenamiento_ml.iloc[indices_train].copy()
        X_val = df_entrenamiento_ml.iloc[indices_val].copy()

        anio = int(X_val["source_year"].iloc[0])
        y_train = X_train["objetivo_ml"].astype(int)

        if preparar is not None:
            X_train, X_val = preparar(X_train, X_val, anio)

        with warnings.catch_warnings(record=True) as avisos:
            warnings.simplefilter("always", ConvergenceWarning)

            ajuste = clone(modelo)
            ajuste.fit(X_train, y_train)

        advertencia = any(
            issubclass(aviso.category, ConvergenceWarning)
            for aviso in avisos
        )

        iteraciones = int(
            ajuste.named_steps["clasificador"].n_iter_.max()
        )

        revision_convergencia.append({
            "modelo": nombre,
            "anio": anio,
            "iteraciones": iteraciones,
            "advertencia": advertencia
        })

tabla_convergencia = pd.DataFrame(revision_convergencia)

display(
    tabla_convergencia.groupby("modelo").agg(
        max_iteraciones=("iteraciones", "max"),
        advertencias=("advertencia", "sum")
    )
)

print(
    "Ajustes con advertencia:",
    tabla_convergencia["advertencia"].sum(),
    "de",
    len(tabla_convergencia)
)

,max_iteraciones,advertencias
modelo,,
Control,11,0
Local cuadrático,12,0
Local lineal,11,0
PCA 2 cuadrático,12,0
PCA 2 lineal,11,0
PCA 3 cuadrático,12,0
PCA 3 lineal,10,0


Ajustes con advertencia: 0 de 35


### 12.6 Selección del modelo

In [26]:
# Modelos que consideraremos para la evaluación final

candidatos_finales = [
    "Control",
    "Local cuadrático",
    "PCA 2 cuadrático"
]

tabla_seleccion = resumen_modelos.loc[
    (resumen_modelos["muestra"] == "Estándar")
    & (resumen_modelos["modelo"].isin(candidatos_finales)),
    [
        "modelo",
        "auc_medio",
        "ap_medio",
        "balanceada_media",
        "mejora_auc",
        "anios_mejora"
    ]
].sort_values("auc_medio", ascending=False)

assert len(tabla_seleccion) == 3
display(tabla_seleccion.round(4))

,modelo,auc_medio,ap_medio,balanceada_media,mejora_auc,anios_mejora
8,Local cuadrático,0.7845,0.7919,0.7106,0.0297,5
10,PCA 2 cuadrático,0.7815,0.7889,0.7069,0.0267,5
7,Control,0.7548,0.7616,0.6820,0.0000,0


### 12.6 Protocolo de evaluación final

In [27]:
# Registrar las decisiones tomadas con 2019–2024

import json

clasificador_final = modelo_cinematica_cuadratica.named_steps["clasificador"]

protocolo_ml = {
    "version": "pre2025_v1",
    "entrenamiento": "2019-2024",
    "prueba_reservada": 2025,
    "muestra_principal": "estandar",
    "objetivo": "objetivo_ml",
    "clase_positiva": "tenue",
    "limite_brillante": -0.89,
    "limite_tenue": -0.07,

    "modelo_principal": "local_cuadratico_diagonal",
    "modelo_control": "control",
    "modelo_alternativo": "pca_2_cuadratico_diagonal",

    "variables_fase": variables_fase,
    "grado_fase": 2,
    "variables_calidad": variables_calidad,
    "variables_cinematica": variables_cinematica,

    "regresion": {
        "C": float(clasificador_final.C),
        "solver": clasificador_final.solver,
        "max_iter": int(clasificador_final.max_iter),
    },

    "umbral_clasificacion": 0.5,
    "metrica_principal": "roc_auc",
    "metricas_secundarias": [
        "average_precision",
        "exactitud_balanceada"
    ],

    "validacion_interna": "temporal_creciente_condicional",
    "evaluacion_estricta": "subconjunto_del_modelo_estandar",
    "ajustar_pca_con": "todas_las_trayectorias_elegibles_del_entrenamiento",
    "numero_pc_alternativa": 2,
    "reajustar_con_2025": False,
}

ruta_config = raiz_proyecto / "configs" / "protocolo_ml_pre2025.json"
ruta_config.parent.mkdir(parents=True, exist_ok=True)

with open(ruta_config, "w", encoding="utf-8") as archivo:
    json.dump(protocolo_ml, archivo, indent=2, ensure_ascii=False)

print("Protocolo guardado:", ruta_config)

Protocolo guardado: C:\Users\pabpg\Documents\ignitia\Investigacion_geminidas\configs\protocolo_ml_pre2025.json


### Comprobaciones finales

In [28]:
# Diferencias por año entre los modelos seleccionados

metrica_ap = ("average_precision")

metricas_revisar = ["roc_auc", metrica_ap, "exactitud_balanceada"]

referencias = {
    "Control": metricas_control,
    "PCA 2 cuadrático": metricas_pca_2_cuadratico,
    "PCA 3 cuadrático": metricas_pca_3_cuadratico,
}

principal = metricas_cinematica_cuadratica.query(
    "muestra == 'Estándar'"
)

filas_comparacion = []

for nombre, tabla in referencias.items():
    referencia = tabla.query("muestra == 'Estándar'")

    comparacion = principal.merge(
        referencia,
        on="anio_validacion",
        suffixes=("_local", "_referencia"),
        validate="one_to_one"
    )

    for _, fila in comparacion.iterrows():
        filas_comparacion.append({
            "comparacion": f"Local - {nombre}",
            "anio": int(fila["anio_validacion"]),
            "delta_auc": (
                fila["roc_auc_local"] - fila["roc_auc_referencia"]
            ),
            "delta_ap": (
                fila[f"{metrica_ap}_local"]
                - fila[f"{metrica_ap}_referencia"]
            ),
            "delta_balanceada": (
                fila["exactitud_balanceada_local"]
                - fila["exactitud_balanceada_referencia"]
            )
        })

tabla_diferencias = pd.DataFrame(filas_comparacion)

display(tabla_diferencias.round(4))

display(
    tabla_diferencias.groupby("comparacion")[
        ["delta_auc", "delta_ap", "delta_balanceada"]
    ].mean().round(4)
)

,comparacion,anio,delta_auc,delta_ap,delta_balanceada
0,Local - Control,2020,0.0447,0.0377,0.0417
1,Local - Control,2021,0.0105,0.0192,0.0178
2,Local - Control,2022,0.0432,0.0528,0.0401
3,Local - Control,2023,0.0314,0.0174,0.0235
4,Local - Control,2024,0.0187,0.0244,0.0201
5,Local - PCA 2 cuadrático,2020,0.0030,0.0009,-0.0013
6,Local - PCA 2 cuadrático,2021,0.0033,0.0019,0.0063
7,Local - PCA 2 cuadrático,2022,0.0009,0.0024,0.0079
8,Local - PCA 2 cuadrático,2023,0.0039,0.0029,0.0070
9,Local - PCA 2 cuadrático,2024,0.0037,0.0067,-0.0016


,delta_auc,delta_ap,delta_balanceada
comparacion,,,
Local - Control,0.0297,0.0303,0.0286
Local - PCA 2 cuadrático,0.0030,0.0030,0.0037
Local - PCA 3 cuadrático,0.0010,0.0006,0.0019


In [29]:
# Coeficientes del modelo principal en cada pliegue temporal

nombres_terminos = (
    ["fase", "fase_cuadrada"]
    + variables_calidad
    + variables_cinematica
    + [f"{v}^2" for v in variables_cinematica]
)

filas_coeficientes = []

for indices_train, indices_val in pliegues_temporales_estandar:

    datos_train = df_entrenamiento_ml.iloc[indices_train]
    datos_val = df_entrenamiento_ml.iloc[indices_val]

    anio = int(datos_val["source_year"].iloc[0])

    ajuste = clone(modelo_cinematica_cuadratica)

    ajuste.fit(
        datos_train,
        datos_train["objetivo_ml"].astype(int)
    )

    coeficientes = ajuste.named_steps["clasificador"].coef_[0]

    assert len(coeficientes) == len(nombres_terminos)

    for termino, coeficiente in zip(nombres_terminos, coeficientes):
        filas_coeficientes.append({
            "anio_validacion": anio,
            "termino": termino,
            "coeficiente": coeficiente
        })

tabla_coeficientes = pd.DataFrame(filas_coeficientes)

terminos_cinematica = (
    variables_cinematica
    + [f"{v}^2" for v in variables_cinematica]
)

tabla_estabilidad = tabla_coeficientes.pivot(
    index="anio_validacion",
    columns="termino",
    values="coeficiente"
)[terminos_cinematica]

display(tabla_estabilidad.round(4))

termino_paralelo_cuadrado = f"{variables_cinematica[0]}^2"

print(
    "Signos del coeficiente paralelo cuadrático:",
    np.sign(tabla_estabilidad[termino_paralelo_cuadrado]).tolist()
)

termino,delta_v_paralela_km_s,delta_v_lambda_km_s,delta_v_beta_km_s,delta_v_paralela_km_s^2,delta_v_lambda_km_s^2,delta_v_beta_km_s^2
anio_validacion,,,,,,
2020,-0.8708,-0.2792,-0.2930,0.8122,-0.2375,-0.3455
2021,-0.9854,-0.1981,-0.0586,1.0109,-0.1345,-0.2569
2022,-0.7797,-0.2646,-0.0269,0.9799,-0.1953,-0.2201
2023,-0.7095,-0.2049,-0.0815,0.6198,-0.0481,-0.1713
2024,-0.6433,-0.2088,-0.0318,0.4588,-0.0300,-0.1483


Signos del coeficiente paralelo cuadrático: [1.0, 1.0, 1.0, 1.0, 1.0]


In [30]:
# Revisar las direcciones PCA aprendidas en cada pliegue

from sklearn.decomposition import PCA

datos_globales = df_entrenamiento.loc[
    df_entrenamiento["usar_pca"],
    variables_cinematica
]

pca_referencia = PCA(n_components=3).fit(datos_globales)

filas_estabilidad_pca = []

for anio in range(2020, 2025):

    datos_anteriores = df_entrenamiento.loc[
        df_entrenamiento["usar_pca"]
        & (df_entrenamiento["source_year"] < anio),
        variables_cinematica
    ]

    pca_anio = PCA(n_components=3).fit(datos_anteriores)

    # Comparar direcciones sin depender del signo arbitrario
    similitud = np.abs(
        pca_anio.components_ @ pca_referencia.components_.T
    )

    filas_estabilidad_pca.append({
        "anio_validacion": anio,
        "n_ajuste": len(datos_anteriores),
        "PC1_var_%": 100 * pca_anio.explained_variance_ratio_[0],
        "PC2_var_%": 100 * pca_anio.explained_variance_ratio_[1],
        "PC3_var_%": 100 * pca_anio.explained_variance_ratio_[2],
        "similitud_PC1": similitud[0, 0],
        "PC2_mas_parecida_a": f"PC{similitud[1].argmax() + 1}",
        "PC3_mas_parecida_a": f"PC{similitud[2].argmax() + 1}"
    })

tabla_estabilidad_pca = pd.DataFrame(filas_estabilidad_pca)

display(tabla_estabilidad_pca.round(4))

,anio_validacion,n_ajuste,PC1_var_%,PC2_var_%,PC3_var_%,similitud_PC1,PC2_mas_parecida_a,PC3_mas_parecida_a
0,2020,430,74.3174,14.4956,11.1870,0.9996,PC2,PC3
1,2021,1432,79.4862,11.4734,9.0405,0.9991,PC2,PC3
2,2022,3336,80.8725,11.1273,8.0002,0.9990,PC2,PC3
3,2023,7048,80.4657,11.6631,7.8712,1.0000,PC2,PC3
4,2024,10445,79.8398,12.0573,8.1029,1.0000,PC2,PC3


### 12.8 Sensibilidad a extremos fotométricos

In [31]:
# Corte del 1 % más brillante, definido en entrenamiento

magnitudes_brillantes = df_entrenamiento_ml.loc[
    df_entrenamiento_ml["objetivo_ml"] == 0,
    "magnitud_analisis"
]

corte_extremos = float(magnitudes_brillantes.quantile(0.01))

n_excluidos = (magnitudes_brillantes < corte_extremos).sum()

print("Corte de magnitud:", round(corte_extremos, 4))
print("Brillantes excluidos:", n_excluidos)
print("Brillantes originales:", len(magnitudes_brillantes))

Corte de magnitud: -3.3701
Brillantes excluidos: 43
Brillantes originales: 4300


In [32]:
# Evaluar el control y el modelo principal sin los extremos brillantes

predicciones_revisar = {
    "Control": predicciones_control,
    "Local cuadrático": predicciones_cinematica_cuadratica
}

magnitudes_por_id = df_entrenamiento_ml[
    ["id_fila_origen", "magnitud_analisis"]
]

filas_sensibilidad = []

for nombre, predicciones in predicciones_revisar.items():

    datos = predicciones.merge(
        magnitudes_por_id,
        on="id_fila_origen",
        validate="one_to_one"
    )

    for anio, grupo in datos.groupby("anio_validacion"):

        conjuntos = {
            "Original": grupo,
            "Sin extremos": grupo.loc[
                grupo["magnitud_analisis"] >= corte_extremos
            ]
        }

        for seleccion, muestra in conjuntos.items():

            y = muestra["objetivo_real"].astype(int)
            p = muestra["probabilidad_tenue"]
            pred = muestra["prediccion"].astype(int)

            assert y.nunique() == 2

            filas_sensibilidad.append({
                "modelo": nombre,
                "anio": anio,
                "seleccion": seleccion,
                "n": len(muestra),
                "roc_auc": roc_auc_score(y, p),
                "average_precision": average_precision_score(y, p),
                "exactitud_balanceada": balanced_accuracy_score(y, pred)
            })

tabla_sensibilidad = pd.DataFrame(filas_sensibilidad)

local = tabla_sensibilidad.query(
    "modelo == 'Local cuadrático'"
)

control = tabla_sensibilidad.query(
    "modelo == 'Control'"
)

comparacion_extremos = local.merge(
    control,
    on=["anio", "seleccion"],
    suffixes=("_local", "_control"),
    validate="one_to_one"
)

for metrica in [
    "roc_auc", "average_precision", "exactitud_balanceada"
]:
    comparacion_extremos[f"mejora_{metrica}"] = (
        comparacion_extremos[f"{metrica}_local"]
        - comparacion_extremos[f"{metrica}_control"]
    )

display(
    comparacion_extremos[
        [
            "anio", "seleccion", "n_local",
            "mejora_roc_auc",
            "mejora_average_precision",
            "mejora_exactitud_balanceada"
        ]
    ].round(4)
)

,anio,seleccion,n_local,mejora_roc_auc,mejora_average_precision,mejora_exactitud_balanceada
0,2020,Original,703,0.0447,0.0377,0.0417
1,2020,Sin extremos,702,0.0444,0.0374,0.0418
2,2021,Original,1319,0.0105,0.0192,0.0178
3,2021,Sin extremos,1315,0.0114,0.0200,0.0181
4,2022,Original,2433,0.0432,0.0528,0.0401
5,2022,Sin extremos,2405,0.0432,0.0518,0.0399
6,2023,Original,2258,0.0314,0.0174,0.0235
7,2023,Sin extremos,2254,0.0315,0.0174,0.0236
8,2024,Original,1645,0.0187,0.0244,0.0201
9,2024,Sin extremos,1639,0.0186,0.0243,0.0197


### 12.9 Entrenamiento final con 2019-2024

In [33]:
# Ajustar los modelos finales utilizando todo el entrenamiento

from joblib import dump, load
from sklearn.decomposition import PCA

assert df_entrenamiento_ml["source_year"].between(2019, 2024).all()

y_final = df_entrenamiento_ml["objetivo_ml"].astype(int)

# Control y modelo local cuadrático
control_final = clone(modelo_control)
control_final.fit(df_entrenamiento_ml, y_final)

local_final = clone(modelo_cinematica_cuadratica)
local_final.fit(df_entrenamiento_ml, y_final)

# PCA global: todos los meteoros elegibles de entrenamiento
datos_pca_final = df_entrenamiento.loc[
    df_entrenamiento["usar_pca"],
    variables_cinematica
]

pca_final = PCA(n_components=3)
pca_final.fit(datos_pca_final)

# Proyectar las filas utilizadas por la regresión
datos_ml_pca = df_entrenamiento_ml.copy()
coordenadas = pca_final.transform(
    datos_ml_pca[variables_cinematica]
)

datos_ml_pca["PC1"] = coordenadas[:, 0]
datos_ml_pca["PC2"] = coordenadas[:, 1]

# Modelo reducido con PCA
pca2_final = clone(modelo_pca_2_cuadratico)
pca2_final.fit(datos_ml_pca, y_final)

print("Meteoros para regresión:", len(df_entrenamiento_ml))
print("Meteoros para PCA:", len(datos_pca_final))

for nombre, modelo in [
    ("Control", control_final),
    ("Local cuadrático", local_final),
    ("PCA 2 cuadrático", pca2_final),
]:
    print(
        nombre,
        "- iteraciones:",
        modelo.named_steps["clasificador"].n_iter_[0]
    )

Meteoros para regresión: 8632
Meteoros para PCA: 12904
Control - iteraciones: 11
Local cuadrático - iteraciones: 12
PCA 2 cuadrático - iteraciones: 10


In [34]:
# Guardar modelos y transformaciones aprendidas

ruta_modelos = (
    raiz_proyecto / "reports" / "models"
    / "modelos_pre2025.joblib"
)

ruta_modelos.parent.mkdir(parents=True, exist_ok=True)

paquete_modelos = {
    "control": control_final,
    "local_cuadratico": local_final,
    "pca_2_cuadratico": pca2_final,
    "pca_global": pca_final,
    "variables_cinematica": variables_cinematica,
}

dump(paquete_modelos, ruta_modelos)

# Comprobar que el archivo conserva las predicciones
modelos_recuperados = load(ruta_modelos)

for nombre, datos in [
    ("control", df_entrenamiento_ml),
    ("local_cuadratico", df_entrenamiento_ml),
    ("pca_2_cuadratico", datos_ml_pca),
]:
    probabilidades_originales = (
        paquete_modelos[nombre].predict_proba(datos)[:, 1]
    )

    probabilidades_recuperadas = (
        modelos_recuperados[nombre].predict_proba(datos)[:, 1]
    )

    np.testing.assert_allclose(
        probabilidades_originales,
        probabilidades_recuperadas,
        rtol=0,
        atol=1e-12
    )

print("Modelos guardados:", ruta_modelos)
print("Comprobación de recuperación: correcta")

Modelos guardados: C:\Users\pabpg\Documents\ignitia\Investigacion_geminidas\reports\models\modelos_pre2025.joblib
Comprobación de recuperación: correcta


In [35]:
# Completar el protocolo con las decisiones finales

protocolo_ml["sensibilidad_fotometrica"] = {
    "tipo": "exclusion_de_extremos_en_evaluacion",
    "variable": "magnitud_analisis",
    "percentil": 0.01,
    "poblacion_para_corte": "brillantes_2019_2024",
    "corte": float(corte_extremos),
    "excluidos_en_desarrollo": int(n_excluidos),
    "reentrenamiento": False,
}

protocolo_ml["filas_entrenamiento_ml"] = int(len(df_entrenamiento_ml))
protocolo_ml["filas_ajuste_pca"] = int(len(datos_pca_final))

protocolo_ml["transformacion_cinematica"] = (
    "componentes locales, cuadrados y StandardScaler"
)

protocolo_ml["transformacion_pca_alternativa"] = (
    "PCA ajustado con todas las trayectorias elegibles "
    "de 2019-2024; utilizar PC1 y PC2"
)

protocolo_ml["archivo_modelos"] = str(
    ruta_modelos.relative_to(raiz_proyecto)
)

protocolo_ml["sha256_modelos"] = hashlib.sha256(
    ruta_modelos.read_bytes()
).hexdigest()

with open(ruta_config, "w", encoding="utf-8") as archivo:
    json.dump(protocolo_ml, archivo, indent=2, ensure_ascii=False)

print("Protocolo actualizado:", ruta_config)
print("Corte de magnitud:", protocolo_ml["sensibilidad_fotometrica"]["corte"])
print("SHA-256:", protocolo_ml["sha256_modelos"])

Protocolo actualizado: C:\Users\pabpg\Documents\ignitia\Investigacion_geminidas\configs\protocolo_ml_pre2025.json
Corte de magnitud: -3.3701
SHA-256: 44bf767d6b0a803d44b9ca04a6a74acfc60004db050ccc83cd5cb92c6b58d78e
